In [1]:
"""
Bluestock MF - EDA
==================

Purpose:
    Exploratory Data Analysis for the 10 numbered assignment datasets.

This script does NOT invent cleaning rules.

It:
    1. Loads all 10 numbered CSV files.
    2. Displays shapes and columns.
    3. Checks data types.
    4. Checks missing values.
    5. Checks duplicate rows.
    6. Produces numeric summaries.
    7. Examines important categorical columns.
    8. Validates AMFI-code relationships.
    9. Examines date ranges where applicable.
   10. Saves EDA reports to reports/eda/.

Expected structure:

Bluestock internship/
|
|-- data/
|   |-- raw/
|       |-- 01_fund_master.csv
|       |-- 02_nav_history.csv
|       |-- 03_aum_by_fund_house.csv
|       |-- 04_monthly_sip_inflows.csv
|       |-- 05_category_inflows.csv
|       |-- 06_industry_folio_count.csv
|       |-- 07_scheme_performance.csv
|       |-- 08_investor_transactions.csv
|       |-- 09_portfolio_holdings.csv
|       |-- 10_benchmark_indices.csv
|
|-- reports/
|   |-- eda/
|
|-- day1_eda.py
"""

from pathlib import Path
import re
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")


# ================================================================
# 1. PROJECT PATH
# ================================================================

def find_project_root():

    candidates = [
        Path.cwd(),
        Path(__file__).resolve().parent
        if "__file__" in globals()
        else Path.cwd(),
        Path(r"C:\Users\javee\Bluestock internship"),
    ]

    checked = set()

    for candidate in candidates:

        if candidate is None:
            continue

        candidate = candidate.resolve()

        if str(candidate) in checked:
            continue

        checked.add(str(candidate))

        if (candidate / "data" / "raw").exists():
            return candidate

    return Path.cwd()


BASE = find_project_root()

RAW_PATH = BASE / "data" / "raw"

REPORT_PATH = BASE / "reports" / "eda"

REPORT_PATH.mkdir(
    parents=True,
    exist_ok=True
)


# ================================================================
# 2. FILES
# ================================================================

EXPECTED_FILES = [
    "01_fund_master.csv",
    "02_nav_history.csv",
    "03_aum_by_fund_house.csv",
    "04_monthly_sip_inflows.csv",
    "05_category_inflows.csv",
    "06_industry_folio_count.csv",
    "07_scheme_performance.csv",
    "08_investor_transactions.csv",
    "09_portfolio_holdings.csv",
    "10_benchmark_indices.csv",
]


# ================================================================
# 3. HELPERS
# ================================================================

def normalize_column(column):

    return (
        str(column)
        .strip()
        .lower()
        .replace(" ", "")
        .replace("_", "")
        .replace("-", "")
        .replace(".", "")
    )


def find_column(
    df,
    aliases
):

    lookup = {
        normalize_column(column): column
        for column in df.columns
    }

    for alias in aliases:

        key = normalize_column(alias)

        if key in lookup:
            return lookup[key]

    return None


def print_section(title):

    print()
    print("=" * 80)
    print(title)
    print("=" * 80)


def safe_numeric_conversion(series):

    return pd.to_numeric(
        series,
        errors="coerce"
    )


def detect_date_column(df):

    candidates = [
        "date",
        "nav_date",
        "transaction_date",
        "txn_date",
        "as_of_date",
        "performance_date",
        "month",
        "reporting_date"
    ]

    return find_column(
        df,
        candidates
    )


# ================================================================
# 4. LOAD DATASETS
# ================================================================

def load_datasets():

    print_section(
        "BLUESTOCK MF - DATA INGESTION / EDA"
    )

    print(
        f"Project root : {BASE}"
    )

    print(
        f"Raw data     : {RAW_PATH}"
    )

    print(
        f"EDA reports  : {REPORT_PATH}"
    )

    if not RAW_PATH.exists():

        raise FileNotFoundError(
            f"Raw data directory not found:\n{RAW_PATH}"
        )

    datasets = {}

    print()

    for filename in EXPECTED_FILES:

        path = RAW_PATH / filename

        if not path.exists():

            print(
                f"[MISSING] {filename}"
            )

            continue

        print(
            f"[LOADING] {filename}"
        )

        df = pd.read_csv(path)

        datasets[filename] = df

        print(
            f"  Rows    : {len(df):,}"
        )

        print(
            f"  Columns : {len(df.columns)}"
        )

    print()

    print(
        f"Loaded {len(datasets)} / "
        f"{len(EXPECTED_FILES)} datasets."
    )

    return datasets


# ================================================================
# 5. DATASET OVERVIEW
# ================================================================

def dataset_overview(datasets):

    print_section(
        "1. DATASET OVERVIEW"
    )

    rows = []

    for filename, df in datasets.items():

        rows.append(
            {
                "file": filename,
                "rows": len(df),
                "columns": len(df.columns),
                "duplicate_rows": int(
                    df.duplicated().sum()
                ),
                "missing_cells": int(
                    df.isna().sum().sum()
                )
            }
        )

    overview = pd.DataFrame(rows)

    print(
        overview.to_string(
            index=False
        )
    )

    overview.to_csv(
        REPORT_PATH /
        "dataset_overview.csv",
        index=False
    )

    return overview


# ================================================================
# 6. COLUMN INFORMATION
# ================================================================

def column_information(datasets):

    print_section(
        "2. COLUMN INFORMATION"
    )

    all_columns = []

    for filename, df in datasets.items():

        print()
        print("-" * 80)
        print(filename)
        print("-" * 80)

        for column in df.columns:

            dtype = str(
                df[column].dtype
            )

            missing = int(
                df[column].isna().sum()
            )

            missing_pct = (
                missing / len(df) * 100
                if len(df) > 0
                else 0
            )

            unique = int(
                df[column].nunique(
                    dropna=True
                )
            )

            print(
                f"{column:35s} "
                f"dtype={dtype:15s} "
                f"missing={missing:8,d} "
                f"missing%={missing_pct:7.2f} "
                f"unique={unique:10,d}"
            )

            all_columns.append(
                {
                    "file": filename,
                    "column": column,
                    "dtype": dtype,
                    "missing_count": missing,
                    "missing_pct": round(
                        missing_pct,
                        2
                    ),
                    "unique_values": unique
                }
            )

    result = pd.DataFrame(
        all_columns
    )

    result.to_csv(
        REPORT_PATH /
        "column_information.csv",
        index=False
    )

    return result


# ================================================================
# 7. NUMERIC SUMMARIES
# ================================================================

def numeric_summaries(datasets):

    print_section(
        "3. NUMERIC COLUMN SUMMARIES"
    )

    summaries = []

    for filename, df in datasets.items():

        numeric_df = df.select_dtypes(
            include=np.number
        )

        if numeric_df.empty:

            continue

        print()
        print("-" * 80)
        print(filename)
        print("-" * 80)

        summary = numeric_df.describe().T

        print(
            summary.to_string()
        )

        for column in summary.index:

            row = summary.loc[column]

            summaries.append(
                {
                    "file": filename,
                    "column": column,
                    "count": row.get(
                        "count",
                        np.nan
                    ),
                    "mean": row.get(
                        "mean",
                        np.nan
                    ),
                    "std": row.get(
                        "std",
                        np.nan
                    ),
                    "min": row.get(
                        "min",
                        np.nan
                    ),
                    "25%": row.get(
                        "25%",
                        np.nan
                    ),
                    "50%": row.get(
                        "50%",
                        np.nan
                    ),
                    "75%": row.get(
                        "75%",
                        np.nan
                    ),
                    "max": row.get(
                        "max",
                        np.nan
                    )
                }
            )

    result = pd.DataFrame(
        summaries
    )

    result.to_csv(
        REPORT_PATH /
        "numeric_summaries.csv",
        index=False
    )

    return result


# ================================================================
# 8. CATEGORICAL EXPLORATION
# ================================================================

def categorical_exploration(datasets):

    print_section(
        "4. IMPORTANT CATEGORICAL COLUMNS"
    )

    output = []

    important_aliases = {
        "fund_house": [
            "fund_house",
            "amc",
            "amc_name"
        ],

        "category": [
            "category",
            "scheme_category"
        ],

        "sub_category": [
            "sub_category",
            "subcategory",
            "scheme_type"
        ],

        "risk": [
            "risk_category",
            "risk_grade",
            "risk"
        ],

        "transaction_type": [
            "transaction_type",
            "txn_type"
        ],

        "kyc_status": [
            "kyc_status",
            "kyc"
        ],

        "state": [
            "state",
            "investor_state"
        ]
    }

    for filename, df in datasets.items():

        for label, aliases in important_aliases.items():

            column = find_column(
                df,
                aliases
            )

            if column is None:
                continue

            print()
            print(
                f"{filename} -> {column}"
            )

            counts = (
                df[column]
                .astype(str)
                .value_counts(
                    dropna=False
                )
                .head(20)
            )

            print(
                counts.to_string()
            )

            for value, count in counts.items():

                output.append(
                    {
                        "file": filename,
                        "column": column,
                        "value": value,
                        "count": int(count)
                    }
                )

    result = pd.DataFrame(
        output
    )

    result.to_csv(
        REPORT_PATH /
        "categorical_exploration.csv",
        index=False
    )

    return result


# ================================================================
# 9. DATE EXPLORATION
# ================================================================

def date_exploration(datasets):

    print_section(
        "5. DATE RANGE EXPLORATION"
    )

    output = []

    for filename, df in datasets.items():

        column = detect_date_column(
            df
        )

        if column is None:
            continue

        parsed = pd.to_datetime(
            df[column],
            errors="coerce",
            dayfirst=True
        )

        valid = parsed.dropna()

        if valid.empty:

            print(
                f"{filename}: "
                f"{column} could not be parsed."
            )

            continue

        print()
        print(
            f"{filename}"
        )

        print(
            f"Date column : {column}"
        )

        print(
            f"Minimum     : {valid.min()}"
        )

        print(
            f"Maximum     : {valid.max()}"
        )

        print(
            f"Invalid     : {parsed.isna().sum():,}"
        )

        output.append(
            {
                "file": filename,
                "date_column": column,
                "minimum_date": valid.min(),
                "maximum_date": valid.max(),
                "invalid_dates": int(
                    parsed.isna().sum()
                )
            }
        )

    result = pd.DataFrame(
        output
    )

    result.to_csv(
        REPORT_PATH /
        "date_ranges.csv",
        index=False
    )

    return result


# ================================================================
# 10. FUND MASTER EDA
# ================================================================

def fund_master_eda(datasets):

    filename = "01_fund_master.csv"

    if filename not in datasets:
        return

    print_section(
        "6. FUND MASTER EDA"
    )

    df = datasets[filename]

    print(
        "Columns:"
    )

    for column in df.columns:
        print(
            f" - {column}"
        )

    amfi_column = find_column(
        df,
        [
            "amfi_code",
            "scheme_code",
            "code"
        ]
    )

    if amfi_column:

        print()
        print(
            f"Unique AMFI codes: "
            f"{df[amfi_column].nunique():,}"
        )

        duplicates = (
            df[amfi_column]
            .duplicated()
            .sum()
        )

        print(
            f"Duplicate AMFI codes: "
            f"{duplicates:,}"
        )

    for label, aliases in {
        "Fund houses": [
            "fund_house",
            "amc",
            "amc_name"
        ],

        "Categories": [
            "category",
            "scheme_category"
        ],

        "Sub-categories": [
            "sub_category",
            "subcategory",
            "scheme_type"
        ],

        "Risk": [
            "risk_category",
            "risk_grade",
            "risk"
        ]
    }.items():

        column = find_column(
            df,
            aliases
        )

        if column:

            print()
            print(
                label
            )

            print(
                df[column]
                .value_counts(
                    dropna=False
                )
                .to_string()
            )


# ================================================================
# 11. AMFI CODE VALIDATION
# ================================================================

def amfi_validation(datasets):

    print_section(
        "7. AMFI CODE VALIDATION"
    )

    fund_file = "01_fund_master.csv"

    if fund_file not in datasets:
        return

    fund = datasets[fund_file]

    fund_column = find_column(
        fund,
        [
            "amfi_code",
            "scheme_code",
            "code"
        ]
    )

    if not fund_column:
        print(
            "AMFI code column not found in fund master."
        )
        return

    fund_codes = set(
        fund[fund_column]
        .astype(str)
        .str.strip()
    )

    output = []

    print(
        f"Fund master unique codes: "
        f"{len(fund_codes):,}"
    )

    for filename, df in datasets.items():

        if filename == fund_file:
            continue

        column = find_column(
            df,
            [
                "amfi_code",
                "scheme_code",
                "code"
            ]
        )

        if column is None:
            continue

        codes = set(
            df[column]
            .astype(str)
            .str.strip()
        )

        matching = (
            fund_codes &
            codes
        )

        only_in_dataset = (
            codes -
            fund_codes
        )

        match_rate = (
            len(matching)
            /
            len(codes)
            * 100
            if codes
            else 0
        )

        print()
        print(
            f"{filename}"
        )

        print(
            f"  Code column       : {column}"
        )

        print(
            f"  Unique codes      : {len(codes):,}"
        )

        print(
            f"  Matching codes    : {len(matching):,}"
        )

        print(
            f"  Not in fund master: "
            f"{len(only_in_dataset):,}"
        )

        print(
            f"  Match rate        : "
            f"{match_rate:.2f}%"
        )

        output.append(
            {
                "file": filename,
                "amfi_column": column,
                "unique_codes": len(codes),
                "matching_codes": len(matching),
                "not_in_fund_master": len(
                    only_in_dataset
                ),
                "match_rate_pct": round(
                    match_rate,
                    2
                )
            }
        )

    result = pd.DataFrame(
        output
    )

    result.to_csv(
        REPORT_PATH /
        "amfi_code_validation.csv",
        index=False
    )


# ================================================================
# 12. NAV EXPLORATION
# ================================================================

def nav_eda(datasets):

    filename = "02_nav_history.csv"

    if filename not in datasets:
        return

    print_section(
        "8. NAV HISTORY EDA"
    )

    df = datasets[filename].copy()

    amfi = find_column(
        df,
        [
            "amfi_code",
            "scheme_code",
            "code"
        ]
    )

    date = find_column(
        df,
        [
            "date",
            "nav_date",
            "as_of_date"
        ]
    )

    nav = find_column(
        df,
        [
            "nav",
            "net_asset_value",
            "nav_value"
        ]
    )

    if amfi:

        print(
            f"Unique AMFI codes: "
            f"{df[amfi].nunique():,}"
        )

    if date:

        parsed = pd.to_datetime(
            df[date],
            errors="coerce",
            dayfirst=True
        )

        print(
            f"Invalid dates: "
            f"{parsed.isna().sum():,}"
        )

        valid = parsed.dropna()

        if not valid.empty:

            print(
                f"Date range: "
                f"{valid.min()} "
                f"to "
                f"{valid.max()}"
            )

    if nav:

        numeric_nav = pd.to_numeric(
            df[nav],
            errors="coerce"
        )

        print(
            f"Non-numeric NAV: "
            f"{numeric_nav.isna().sum():,}"
        )

        print(
            f"NAV <= 0: "
            f"{(numeric_nav <= 0).sum():,}"
        )

        print()
        print(
            numeric_nav.describe()
            .to_string()
        )

    if amfi and date:

        duplicate_key = df.duplicated(
            subset=[
                amfi,
                date
            ]
        )

        print(
            f"Duplicate AMFI/date rows: "
            f"{duplicate_key.sum():,}"
        )


# ================================================================
# 13. TRANSACTION EDA
# ================================================================

def transaction_eda(datasets):

    filename = "08_investor_transactions.csv"

    if filename not in datasets:
        return

    print_section(
        "9. INVESTOR TRANSACTION EDA"
    )

    df = datasets[filename].copy()

    type_column = find_column(
        df,
        [
            "transaction_type",
            "txn_type",
            "type"
        ]
    )

    amount_column = find_column(
        df,
        [
            "amount",
            "txn_amount",
            "transaction_amount"
        ]
    )

    kyc_column = find_column(
        df,
        [
            "kyc_status",
            "kyc"
        ]
    )

    investor_column = find_column(
        df,
        [
            "investor_id",
            "folio_no",
            "folio_number",
            "client_id"
        ]
    )

    if type_column:

        print()
        print(
            "Transaction types:"
        )

        print(
            df[type_column]
            .astype(str)
            .value_counts(
                dropna=False
            )
            .to_string()
        )

    if amount_column:

        amount = pd.to_numeric(
            df[amount_column],
            errors="coerce"
        )

        print()
        print(
            f"Non-numeric amounts: "
            f"{amount.isna().sum():,}"
        )

        print(
            f"Amounts <= 0: "
            f"{(amount <= 0).sum():,}"
        )

        print()
        print(
            amount.describe()
            .to_string()
        )

    if kyc_column:

        print()
        print(
            "KYC status:"
        )

        print(
            df[kyc_column]
            .astype(str)
            .str.upper()
            .value_counts(
                dropna=False
            )
            .to_string()
        )

    if investor_column:

        print()
        print(
            f"Unique investors: "
            f"{df[investor_column].nunique():,}"
        )


# ================================================================
# 14. PERFORMANCE EDA
# ================================================================

def performance_eda(datasets):

    filename = "07_scheme_performance.csv"

    if filename not in datasets:
        return

    print_section(
        "10. SCHEME PERFORMANCE EDA"
    )

    df = datasets[filename].copy()

    print(
        "Available columns:"
    )

    for column in df.columns:
        print(
            f" - {column}"
        )

    numeric_candidates = [
        "return_1y",
        "return_1yr_pct",
        "return_3y",
        "return_3yr_pct",
        "return_5y",
        "return_5yr_pct",
        "expense_ratio",
        "aum_crore",
        "aum"
    ]

    for alias in numeric_candidates:

        column = find_column(
            df,
            [alias]
        )

        if column is None:
            continue

        numeric = pd.to_numeric(
            df[column],
            errors="coerce"
        )

        print()
        print(
            f"{column}"
        )

        print(
            f"Non-numeric: "
            f"{numeric.isna().sum():,}"
        )

        print(
            numeric.describe()
            .to_string()
        )

    expense = find_column(
        df,
        [
            "expense_ratio",
            "expense_ratio_pct"
        ]
    )

    if expense:

        expense_numeric = pd.to_numeric(
            df[expense],
            errors="coerce"
        )

        invalid_expense = (
            expense_numeric.notna()
            &
            ~expense_numeric.between(
                0.1,
                2.5
            )
        )

        print()
        print(
            f"Expense ratio outside "
            f"0.1%-2.5%: "
            f"{invalid_expense.sum():,}"
        )


# ================================================================
# 15. AUM EDA
# ================================================================

def aum_eda(datasets):

    filename = "03_aum_by_fund_house.csv"

    if filename not in datasets:
        return

    print_section(
        "11. AUM BY FUND HOUSE EDA"
    )

    df = datasets[filename]

    print(
        "Actual columns:"
    )

    print(
        list(df.columns)
    )

    for alias in [
        "aum_crore",
        "aum_lakh_crore",
        "num_schemes"
    ]:

        column = find_column(
            df,
            [alias]
        )

        if column is None:
            continue

        numeric = pd.to_numeric(
            df[column],
            errors="coerce"
        )

        print()
        print(
            f"{column}"
        )

        print(
            numeric.describe()
            .to_string()
        )

    fund_house = find_column(
        df,
        [
            "fund_house",
            "amc",
            "amc_name"
        ]
    )

    if fund_house:

        print()
        print(
            "Fund houses:"
        )

        print(
            df[fund_house]
            .value_counts(
                dropna=False
            )
            .to_string()
        )

    date_column = find_column(
        df,
        [
            "date",
            "as_of_date"
        ]
    )

    if date_column:

        dates = pd.to_datetime(
            df[date_column],
            errors="coerce",
            dayfirst=True
        )

        valid = dates.dropna()

        if not valid.empty:

            print()
            print(
                f"Date range: "
                f"{valid.min()} "
                f"to "
                f"{valid.max()}"
            )


# ================================================================
# 16. OTHER DATASETS
# ================================================================

def other_dataset_eda(datasets):

    print_section(
        "12. OTHER DATASET EDA"
    )

    excluded = {
        "01_fund_master.csv",
        "02_nav_history.csv",
        "03_aum_by_fund_house.csv",
        "07_scheme_performance.csv",
        "08_investor_transactions.csv"
    }

    for filename, df in datasets.items():

        if filename in excluded:
            continue

        print()
        print("-" * 80)
        print(filename)
        print("-" * 80)

        print(
            f"Shape: {df.shape}"
        )

        print()
        print(
            "Columns:"
        )

        for column in df.columns:
            print(
                f" - {column}: "
                f"{df[column].dtype}"
            )

        print()
        print(
            "Missing values:"
        )

        missing = df.isna().sum()

        missing = missing[
            missing > 0
        ]

        if missing.empty:

            print(
                "None"
            )

        else:

            print(
                missing.to_string()
            )

        print()
        print(
            f"Duplicate rows: "
            f"{df.duplicated().sum():,}"
        )


# ================================================================
# 17. MISSING VALUE REPORT
# ================================================================

def missing_value_report(datasets):

    print_section(
        "13. MISSING VALUE REPORT"
    )

    output = []

    for filename, df in datasets.items():

        for column in df.columns:

            missing = int(
                df[column].isna().sum()
            )

            output.append(
                {
                    "file": filename,
                    "column": column,
                    "missing_count": missing,
                    "total_rows": len(df),
                    "missing_pct": round(
                        (
                            missing / len(df) * 100
                        )
                        if len(df)
                        else 0,
                        2
                    )
                }
            )

    result = pd.DataFrame(
        output
    )

    result = result.sort_values(
        [
            "file",
            "missing_count"
        ],
        ascending=[
            True,
            False
        ]
    )

    result.to_csv(
        REPORT_PATH /
        "missing_value_report.csv",
        index=False
    )

    print(
        result[
            result["missing_count"] > 0
        ].to_string(
            index=False
        )
    )


# ================================================================
# 18. DUPLICATE REPORT
# ================================================================

def duplicate_report(datasets):

    print_section(
        "14. DUPLICATE ROW REPORT"
    )

    output = []

    for filename, df in datasets.items():

        duplicates = int(
            df.duplicated().sum()
        )

        output.append(
            {
                "file": filename,
                "rows": len(df),
                "duplicate_rows": duplicates,
                "duplicate_pct": round(
                    (
                        duplicates / len(df) * 100
                    )
                    if len(df)
                    else 0,
                    2
                )
            }
        )

    result = pd.DataFrame(
        output
    )

    print(
        result.to_string(
            index=False
        )
    )

    result.to_csv(
        REPORT_PATH /
        "duplicate_report.csv",
        index=False
    )


# ================================================================
# 19. MAIN
# ================================================================

def main():

    datasets = load_datasets()

    if not datasets:

        raise RuntimeError(
            "No datasets were loaded."
        )

    dataset_overview(
        datasets
    )

    column_information(
        datasets
    )

    numeric_summaries(
        datasets
    )

    categorical_exploration(
        datasets
    )

    date_exploration(
        datasets
    )

    fund_master_eda(
        datasets
    )

    amfi_validation(
        datasets
    )

    nav_eda(
        datasets
    )

    transaction_eda(
        datasets
    )

    performance_eda(
        datasets
    )

    aum_eda(
        datasets
    )

    other_dataset_eda(
        datasets
    )

    missing_value_report(
        datasets
    )

    duplicate_report(
        datasets
    )

    print_section(
        "EDA COMPLETE"
    )

    print(
        "EDA reports saved to:"
    )

    print(
        REPORT_PATH
    )

    print()
    print(
        "Generated reports:"
    )

    for file in sorted(
        REPORT_PATH.glob("*")
    ):

        print(
            f" - {file.name}"
        )


if __name__ == "__main__":

    main()



BLUESTOCK MF - DATA INGESTION / EDA
Project root : C:\Users\javee\Bluestock internship\Capstone
Raw data     : C:\Users\javee\Bluestock internship\Capstone\data\raw
EDA reports  : C:\Users\javee\Bluestock internship\Capstone\reports\eda

[LOADING] 01_fund_master.csv
  Rows    : 40
  Columns : 15
[LOADING] 02_nav_history.csv
  Rows    : 46,000
  Columns : 3
[LOADING] 03_aum_by_fund_house.csv
  Rows    : 90
  Columns : 5
[LOADING] 04_monthly_sip_inflows.csv
  Rows    : 48
  Columns : 6
[LOADING] 05_category_inflows.csv
  Rows    : 144
  Columns : 3
[LOADING] 06_industry_folio_count.csv
  Rows    : 21
  Columns : 6
[LOADING] 07_scheme_performance.csv
  Rows    : 40
  Columns : 19
[LOADING] 08_investor_transactions.csv
  Rows    : 32,778
  Columns : 13
[LOADING] 09_portfolio_holdings.csv
  Rows    : 322
  Columns : 8
[LOADING] 10_benchmark_indices.csv
  Rows    : 8,050
  Columns : 3

Loaded 10 / 10 datasets.

1. DATASET OVERVIEW


                        file  rows  columns  duplicate_rows  missing_cells
          01_fund_master.csv    40       15               0              0
          02_nav_history.csv 46000        3               0              0
    03_aum_by_fund_house.csv    90        5               0              0
  04_monthly_sip_inflows.csv    48        6               0             12
     05_category_inflows.csv   144        3               0              0
 06_industry_folio_count.csv    21        6               0              0
   07_scheme_performance.csv    40       19               0              0
08_investor_transactions.csv 32778       13               0              0
   09_portfolio_holdings.csv   322        8               0              0
    10_benchmark_indices.csv  8050        3               0              0

2. COLUMN INFORMATION

--------------------------------------------------------------------------------
01_fund_master.csv
---------------------------------------------------


--------------------------------------------------------------------------------
02_nav_history.csv
--------------------------------------------------------------------------------
             count           mean           std          min            25%           50%            75%          max
amfi_code  46000.0  120247.000000  14352.317221  100016.0000  118632.750000  119551.50000  120842.250000  149324.0000
nav        46000.0     269.570265    577.187060      26.1366      69.170425     122.73215     260.338675    4268.5497

--------------------------------------------------------------------------------
03_aum_by_fund_house.csv
--------------------------------------------------------------------------------
                count           mean            std        min         25%        50%         75%        max
aum_lakh_crore   90.0       4.352889       2.734328       1.05       2.525       3.45       5.675       12.5
aum_crore        90.0  435288.888889  273432.827420  10500

              count          mean           std      min        25%       50%         75%       max
close_value  8050.0  18351.950181  14169.995121  1444.13  2819.0625  18392.96  26539.0975  79075.39

4. IMPORTANT CATEGORICAL COLUMNS

01_fund_master.csv -> fund_house
fund_house
SBI Mutual Fund             5
HDFC Mutual Fund            5
ICICI Prudential MF         5
Nippon India MF             5
Kotak Mahindra MF           4
Axis Mutual Fund            4
Aditya Birla Sun Life MF    3
UTI Mutual Fund             3
Mirae Asset MF              3
DSP Mutual Fund             3

01_fund_master.csv -> category
category
Equity    34
Debt       6

01_fund_master.csv -> sub_category
sub_category
Large Cap          14
Mid Cap             7
Small Cap           6
Liquid              3
Gilt                2
Flexi Cap           2
Short Duration      1
Value               1
Index/ETF           1
Index               1
Large & Mid Cap     1
ELSS                1

01_fund_master.csv -> risk_category
risk


02_nav_history.csv
Date column : date
Minimum     : 2022-01-02 00:00:00
Maximum     : 2026-12-05 00:00:00
Invalid     : 27,880

03_aum_by_fund_house.csv
Date column : date
Minimum     : 2022-03-31 00:00:00
Maximum     : 2025-12-31 00:00:00
Invalid     : 0

04_monthly_sip_inflows.csv
Date column : month
Minimum     : 2022-01-01 00:00:00
Maximum     : 2025-12-01 00:00:00
Invalid     : 0

05_category_inflows.csv
Date column : month
Minimum     : 2024-04-01 00:00:00
Maximum     : 2025-03-01 00:00:00
Invalid     : 0

06_industry_folio_count.csv
Date column : month
Minimum     : 2022-01-01 00:00:00
Maximum     : 2025-12-01 00:00:00
Invalid     : 0

08_investor_transactions.csv
Date column : transaction_date
Minimum     : 2024-01-01 00:00:00
Maximum     : 2025-12-05 00:00:00
Invalid     : 19,730

10_benchmark_indices.csv
Date column : date
Minimum     : 2022-01-02 00:00:00
Maximum     : 2026-12-05 00:00:00
Invalid     : 4,879

6. FUND MASTER EDA
Columns:
 - amfi_code
 - fund_house
 - scheme_

Invalid dates: 27,880
Date range: 2022-01-02 00:00:00 to 2026-12-05 00:00:00
Non-numeric NAV: 0
NAV <= 0: 0

count    46000.000000
mean       269.570265
std        577.187060
min         26.136600
25%         69.170425
50%        122.732150
75%        260.338675
max       4268.549700
Duplicate AMFI/date rows: 0

9. INVESTOR TRANSACTION EDA

Transaction types:
transaction_type
SIP           19716
Lumpsum        8095
Redemption     4967

KYC status:
kyc_status
VERIFIED    30146
PENDING      2632

Unique investors: 5,000

10. SCHEME PERFORMANCE EDA
Available columns:
 - amfi_code
 - scheme_name
 - fund_house
 - category
 - plan
 - return_1yr_pct
 - return_3yr_pct
 - return_5yr_pct
 - benchmark_3yr_pct
 - alpha
 - beta
 - sharpe_ratio
 - sortino_ratio
 - std_dev_ann_pct
 - max_drawdown_pct
 - aum_crore
 - expense_ratio_pct
 - morningstar_rating
 - risk_grade

return_1yr_pct
Non-numeric: 0
count    40.000000
mean     14.376000
std       4.883023
min       4.260000
25%      11.735000
50%    

                        file  rows  duplicate_rows  duplicate_pct
          01_fund_master.csv    40               0            0.0
          02_nav_history.csv 46000               0            0.0
    03_aum_by_fund_house.csv    90               0            0.0
  04_monthly_sip_inflows.csv    48               0            0.0
     05_category_inflows.csv   144               0            0.0
 06_industry_folio_count.csv    21               0            0.0
   07_scheme_performance.csv    40               0            0.0
08_investor_transactions.csv 32778               0            0.0
   09_portfolio_holdings.csv   322               0            0.0
    10_benchmark_indices.csv  8050               0            0.0

EDA COMPLETE
EDA reports saved to:
C:\Users\javee\Bluestock internship\Capstone\reports\eda

Generated reports:
 - amfi_code_validation.csv
 - categorical_exploration.csv
 - column_information.csv
 - dataset_overview.csv
 - date_ranges.csv
 - duplicate_report.csv
 - missin